In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Read the dataset Q1_data.csv using read_csv()
import pandas as pd
df = pd.read_csv("/kaggle/input/q1-ka-ai-2026/Q1_data.csv")

In [ ]:
# Task 2: Write your code here:
#Inspect the first few rows using head()
df.head()

In [ ]:
# Task 3: Write your code here:
# Display dataset information using info()
df.info()

In [ ]:
# Task 4: Write your code here:
#Show statistical description using describe()
df.describe()

In [ ]:
# Task 5: Write your code here:
# Plot the target distribution (delivery_time)
df['Delivery_Time'].hist()

In [ ]:
# Task 1: Write your code here:
# Drop the 'Order_ID' column from the data
df_clean = df.drop('Order_ID',axis=1)
df_clean.head()

In [ ]:
# Task 2: Write your code here:
# Handle missing values appropriately (Hint: I guess you want to have a closer look at the columns with missing values :) )


# Drop rows where target (Delivery_time)  - can't predict without them
print(f"Before: {df_clean.shape}")
df_clean = df_clean.dropna(subset=['Delivery_Time'])
print(f"After dropping missing Delivery_Time {df_clean.shape}")



# Fill categorical columns with 'unknown' - missing likely means "not specified"
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df_clean[col] = df_clean[col].fillna('unknown')

df_clean['Courier_Experience_yrs'].fillna(df_clean['Courier_Experience_yrs'].mean())

print(df_clean.isnull().sum())

In [ ]:
# Task 3: Write your code here:
# Check and remove duplicates if any exist
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)


In [ ]:
# Task 4: Write your code here:
# Encode categorical variables if needed (Bonus if used One Hot Encoding)
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
categorical_cols = ['Weather','Time_of_Day','Vehicle_Type','Traffic_Level']
# onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
# oe_cols_encoded = pd.DataFrame(onehot_encoder.fit_transform(df_clean[oe_cols]), columns=onehot_encoder.get_feature_names_out(oe_cols))
# df_clean.head()
# df_clean.drop(oe_cols)

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_clean[col] = le.fit_transform(df_clean[col])
  label_encoders[col] = le

df_clean.head()



In [ ]:
# Task 5: Write your code here:
# Apply feature scaling for all features (Use StandardScaler)
from sklearn.preprocessing import StandardScaler

numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()

In [ ]:
# Task 6: Write your code here:
# Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)
df_clean['Delivery_Time'].hist() # this is the target and is not a categorical one. SO no imbalance

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Delivery_Time",axis = 1)
y = df_clean['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
import numpy as np

all_mae = []
model = RandomForestRegressor(n_estimators=200)
n_splits = 5
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]


  print(f"Training..")
  # Train
  model.fit(X_train, y_train)
  # Predict
  y_pred = model.predict(X_test)
  # Calculate metrics
  mae = mean_absolute_error(y_test, y_pred)
  # Store results
  all_mae.append(mae)

print(np.mean(all_mae))

In [ ]:
import matplotlib.pyplot as plt
# Task 1: Write your code here:
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: